<a href="https://colab.research.google.com/github/SSK166/PestClefSK/blob/work/PestClefSK2026-work-f1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# PestCLEF 2026 — Fixed Pipeline

**Fixes in this version:**
1. Gemini API URL corrected (`v1beta` not `v1`)
2. NER tokenization alignment completely rewritten — this was the root cause of F1=0.000
3. Silver label quality checks added
4. Diagnosis cell to verify labels before training

In [ ]:
%%capture
!pip install -q seqeval transformers datasets accelerate scikit-learn tqdm requests google-generativeai
print('✅ Done')

In [ ]:
pip install rapidfuzz

In [ ]:
from rapidfuzz import fuzz
import re

def normalize(text):
    return re.sub(r'\s+', ' ', text.lower()).strip()

def get_ngrams(text, n):
    words = text.split()
    return [" ".join(words[i:i+n]) for i in range(len(words)-n+1)]

def fuzzy_find_all(term, text, threshold=85):
    term_n = len(term.split())
    candidates = get_ngrams(text, term_n)

    matches = []
    for cand in candidates:
        score = fuzz.ratio(term, cand)
        if score >= threshold:
            matches.append((cand, score))

    return matches

In [ ]:
import os, re, json, time, random, itertools, unicodedata, difflib, copy, csv
from pathlib import Path
from dataclasses import dataclass
from collections import defaultdict, Counter
from typing import Optional, List, Dict, Tuple, Set

import numpy as np
import torch
import torch.nn as nn
import requests
from tqdm.notebook import tqdm

from sklearn.metrics import f1_score as sk_f1
from seqeval.metrics import classification_report as seq_report
from transformers import (
    AutoTokenizer, AutoModel, AutoModelForTokenClassification, AutoConfig,
    DataCollatorForTokenClassification, TrainingArguments, Trainer,
    get_linear_schedule_with_warmup,
)
from torch.optim import AdamW
from datasets import Dataset

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')
if DEVICE == 'cuda':
    print(f'GPU : {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')
else:
    print('⚠️  No GPU — switch to T4 GPU runtime')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

BASE_DIR       = Path('/content/drive/MyDrive/EPOP_documents')
TRAIN_DIR      = BASE_DIR / 'train'
DEV_DIR        = BASE_DIR / 'dev'
TEST_DIR       = BASE_DIR / 'test'
NER_MODEL_DIR  = BASE_DIR / 'models' / 'ner'
REL_MODEL_DIR  = BASE_DIR / 'models' / 'relation'
SUBMISSION_DIR = BASE_DIR / 'submissions'

for d in [NER_MODEL_DIR, REL_MODEL_DIR, SUBMISSION_DIR]:
    d.mkdir(parents=True, exist_ok=True)

for split_dir in [TRAIN_DIR, DEV_DIR, TEST_DIR]:
    n = len(list(split_dir.glob('*.txt'))) if split_dir.exists() else 0
    print(f'  {split_dir.name}: {n} .txt files')

## Schema

In [ ]:
ENTITY_TYPES = ['Pest','Plant','Disease','Vector','Dissemination_pathway','Location','Date']
LABEL_LIST   = ['O'] + [f'{b}-{e}' for e in ENTITY_TYPES for b in ('B','I')]
LABEL2ID     = {l: i for i, l in enumerate(LABEL_LIST)}
ID2LABEL     = {i: l for l, i in LABEL2ID.items()}

PREDICATES = ['NO_RELATION','Affects','Causes','Dispersed_by','Found_on',
               'Located_in','Occurs_on','Transmits']
PRED2ID    = {p: i for i, p in enumerate(PREDICATES)}
ID2PRED    = {i: p for p, i in PRED2ID.items()}
NO_REL_ID  = PRED2ID['NO_RELATION']

VALID_PAIRS = {
    ('Disease','Plant'):['Affects'],
    ('Pest','Disease'):['Causes'],
    ('Disease','Dissemination_pathway'):['Dispersed_by'],
    ('Pest','Dissemination_pathway'):['Dispersed_by'],
    ('Pest','Plant'):['Found_on'],
    ('Vector','Plant'):['Found_on'],
    ('Vector','Dissemination_pathway'):['Found_on'],
    ('Disease','Location'):['Located_in'],
    ('Pest','Location'):['Located_in'],
    ('Plant','Location'):['Located_in'],
    ('Vector','Location'):['Located_in'],
    ('Disease','Date'):['Occurs_on'],
    ('Pest','Date'):['Occurs_on'],
    ('Plant','Date'):['Occurs_on'],
    ('Vector','Date'):['Occurs_on'],
    ('Vector','Disease'):['Transmits'],
    ('Vector','Pest'):['Transmits'],
}

def predicate_normalize(p): return re.sub(r'[\s_]+','',p.lower())

print(f'Labels: {len(LABEL_LIST)} | Predicates: {len(PREDICATES)} | Pairs: {len(VALID_PAIRS)}')

## Data loading

In [ ]:
def _clean(text):
    text = re.sub(r'[ \t]+', ' ', text)
    text = re.sub(r'\n{3,}', '\n\n', text)
    return '\n'.join(l.strip() for l in text.split('\n') if len(l.strip())>2).strip()

def load_split(split_dir, verbose=True):
    split_dir = Path(split_dir)
    meta = {}
    csv_p = split_dir / 'document_metadata.csv'
    if csv_p.exists():
        with open(csv_p, newline='', encoding='utf-8') as f:
            for row in csv.DictReader(f):
                did = (row.get('document_no') or row.get('doc_id') or '').strip()
                if did: meta[did] = (row.get('website') or '').strip()
    docs = []
    for fp in sorted(split_dir.glob('*.txt')):
        try:
            text = _clean(fp.read_text('utf-8', errors='replace'))
            docs.append({'id': fp.stem, 'raw_text': text,
                         'source_url': meta.get(fp.stem,''),
                         'entities':[], 'relations':[]})
        except Exception as e:
            print(f'[WARN] {fp.name}: {e}')
    if verbose:
        avg = int(np.mean([len(d['raw_text']) for d in docs])) if docs else 0
        print(f'✅ [{split_dir.name}] {len(docs)} docs | avg {avg} chars')
    return docs

In [ ]:
!pip install transformers accelerate sentencepiece

## Silver labeling

In [ ]:

def extract_entities(text):
    entities = []
    used_spans = []

    TERMS_SORTED = sorted(TERM_DICT.items(), key=lambda x: -len(x[0]))

    for term, etype in TERMS_SORTED:
        if len(term) < 4:
            continue
        if term in {'the', 'and', 'for', 'with', 'from', 'this', 'that'}:
            continue

        pattern = r'\b' + re.escape(term) + r'\b'

        for m in re.finditer(pattern, text, flags=re.IGNORECASE):
            start, end = m.start(), m.end()

            span_text = text[start:end]

            # 🚫 reject junk spans
            if len(span_text.strip()) < 3:
                continue
            if not any(c.isalpha() for c in span_text):
                continue

            # 🚫 overlap check
            if any(not (end <= s or start >= e) for s, e in used_spans):
                continue

            entities.append({
                'text': span_text,
                'type': etype,
                'start': start,
                'end': end,
                'score': 1.0,
                'canonical_id': term
            })

            used_spans.append((start, end))

    return entities

def extract_relations(text, entities):
    relations = []
    seen = set()

    sentences = re.split(r'(?<=[.!?])\s+', text)

    for sent in sentences:
        sent_start = text.find(sent)
        sent_end = sent_start + len(sent)

        present = [
            e for e in entities
            if e["start"] >= sent_start and e["end"] <= sent_end
        ]

        for e1, e2 in itertools.permutations(present, 2):
            pair = (e1["type"], e2["type"])

            if pair not in PREDICATE_MAP:
                continue

            pred = PREDICATE_MAP[pair]
            key = (e1["text"], pred, e2["text"])

            if key in seen:
                continue

            seen.add(key)

            relations.append({
                "subject": e1["text"],
                "predicate": pred,
                "object": e2["text"],
                "score": min(e1.get("score", 1.0), e2.get("score", 1.0))
            })

    return relations

In [ ]:
# Rule-based fallback
PEST_TERMS = [
    'Phytophthora infestans','Plasmopara viticola','Botrytis cinerea',
    'Fusarium oxysporum','Puccinia striiformis','Alternaria solani',
    'Blumeria graminis','Verticillium dahliae','Xylella fastidiosa',
    'Ralstonia solanacearum','Erwinia amylovora','Bursaphelenchus xylophilus',
    'Meloidogyne incognita','Globodera pallida','Myzus persicae',
    'Bemisia tabaci','Trialeurodes vaporariorum','Frankliniella occidentalis',
    'Spodoptera frugiperda','Helicoverpa armigera','Tuta absoluta',
    'Drosophila suzukii','Bactrocera dorsalis','Ceratitis capitata',
    'fall armyworm','tomato leafminer','spotted wing drosophila',
    'green peach aphid','silverleaf whitefly','western flower thrips',
    'pine wood nematode','medfly','ToBRFV','TYLCV','TSWV','PVY','PPV','HLB',
    'citrus greening','huanglongbing','bacterial wilt','fire blight',
    'gray mold','grey mold','late blight','early blight',
    'powdery mildew','downy mildew','yellow rust','stripe rust',
]
PLANT_TERMS = [
    'tomato','potato','wheat','rice','maize','corn','soybean','grape',
    'grapevine','apple','citrus','orange','lemon','cotton','barley',
    'sunflower','pepper','cucumber','strawberry','olive','peach','plum',
    'banana','coffee','sugar beet','pine',
    'Solanum tuberosum','Solanum lycopersicum','Triticum aestivum',
    'Oryza sativa','Zea mays','Vitis vinifera','Malus domestica',
]
DISEASE_TERMS = [
    'late blight','early blight','powdery mildew','downy mildew',
    'gray mold','grey mold','fire blight','citrus greening',
    'bacterial wilt'
]
VECTOR_TERMS = [
    'aphid','whitefly','thrips','leafhopper','psyllid','mealybug',
    'Myzus persicae','Bemisia tabaci','Frankliniella occidentalis',
]
PATHWAY_TERMS = [
    'infected plant material',
    'contaminated soil',
    'wood packaging',
    'wooden crates',
]
LOCATION_TERMS = [
    'France','Spain','Italy','Germany','USA','United States','China',
    'India','Brazil','Australia','Europe','Africa','Asia','Mediterranean',
    'North America','South America','worldwide','global',
]

TERM_DICT = {}
for t in PEST_TERMS:     TERM_DICT[t] = 'Pest'
for t in PLANT_TERMS:    TERM_DICT[t] = 'Plant'
for t in DISEASE_TERMS:  TERM_DICT[t] = 'Disease'
for t in VECTOR_TERMS:   TERM_DICT[t] = 'Vector'
for t in PATHWAY_TERMS:  TERM_DICT[t] = 'Dissemination_pathway'
for t in LOCATION_TERMS: TERM_DICT[t] = 'Location'



# remove bad terms
BAD_TERMS = {
    'a','an','the','of','to','in','on','for','by','with','and'
}

TERM_DICT = {
    k: v for k, v in TERM_DICT.items()
    if len(k) > 3 and k not in BAD_TERMS
}
TERM_DICT.update({
    'japanese beetle': 'Pest',
    'late blight': 'Disease',
    'early blight': 'Disease',
    'powdery mildew': 'Disease',
})

PREDICATE_MAP = {
    ('Pest','Disease'):'Causes', ('Disease','Plant'):'Affects',
    ('Vector','Pest'):'Transmits', ('Vector','Disease'):'Transmits',
    ('Pest','Dissemination_pathway'):'Dispersed_by',
    ('Disease','Dissemination_pathway'):'Dispersed_by',
    ('Pest','Plant'):'Found_on',
    ('Pest','Location'):'Located_in',('Disease','Location'):'Located_in',
    ('Plant','Location'):'Located_in',('Vector','Location'):'Located_in',
}

def high_quality_silver_label_doc(doc):
    text = doc['raw_text']

    entities = extract_entities(text)
    relations = extract_relations(text, entities)

    doc['entities'] = entities
    doc['relations'] = relations

    return doc

def rule_based_silver_label_doc(doc):
    text = doc["raw_text"]

    entities = extract_entities(text)

    ent_map = {(e["start"], e["end"], e["text"]): e for e in entities}

    relations = []
    seen = set()

    sentences = re.split(r'(?<=[.!?])\s+', text)

    for sent in sentences:
        sent_start = text.find(sent)
        sent_end = sent_start + len(sent)

        present = [
            e for e in ent_map.values()
            if e["start"] >= sent_start and e["end"] <= sent_end
        ]

        for e1, e2 in itertools.combinations(present, 2):
            pair = (e1["type"], e2["type"])

            if pair not in PREDICATE_MAP:
                continue

            pred = PREDICATE_MAP[pair]
            key = (e1["text"], pred, e2["text"])

            if key in seen:
                continue

            seen.add(key)

            relations.append({
                "subject": e1["text"],
                "predicate": pred,
                "object": e2["text"],
                "score": 0.7
            })

    doc["entities"] = list(ent_map.values())
    doc["relations"] = relations

    return doc

In [ ]:
for key,val in TERM_DICT.items():
  print(key,val)

In [ ]:
print("Sample TERM_DICT entries:")
for i, (k,v) in enumerate(list(TERM_DICT.items())[:20]):
    print(k, "->", v)

In [ ]:
def filter_relations(rels):
    """
    Cleans invalid / noisy predicted relations before submission.
    This alone boosts leaderboard score.
    """

    cleaned = []
    for r in rels:
        s = r["subject"].strip()
        o = r["object"].strip()

        # remove garbage tokens
        bad_tokens = {"", "recently", "area", "province", "country", "world", "international"}

        if len(s) < 2 or len(o) < 2:
            continue
        if s.lower() in bad_tokens or o.lower() in bad_tokens:
            continue
        if s == o:
            continue

        cleaned.append(r)

    return cleaned

In [ ]:
def filter_entities(entities):
    """
    Keeps only meaningful entities for relation extraction.
    """

    cleaned = []
    for e in entities:
        text = e["text"].strip()

        # remove bad entities
        if len(text) < 2:
            continue
        if text.lower() in {"planting", "recently", "area", "country"}:
            continue
        if text.isdigit():
            continue

        cleaned.append(e)

    return cleaned

## NER — Fixed tokenization

**Root cause of F1=0.000:** The old `silver_doc_to_hf` used `re.finditer(r'\S+', text)` to split into tokens, then mapped char-level labels to those tokens. But the HuggingFace `Trainer` re-tokenizes those word strings using BioBERT's WordPiece tokenizer. The word boundaries from regex don't match BioBERT's word boundaries, so the label-to-subtoken alignment is completely wrong — every token gets label 0 (= `O`).

**The fix:** use `word_ids()` properly. Pass the raw words to the tokenizer with `is_split_into_words=True`, then use `word_ids()` to align the original word-level BIO labels to subword tokens. The first subword of a word gets the real label; continuation subwords get `-100` (ignored in loss).

In [ ]:
from seqeval.metrics import f1_score, precision_score, recall_score

BASE_NER_MODEL = 'dmis-lab/biobert-base-cased-v1.2'


def doc_to_hf_fixed(doc: dict, tokenizer):
    text = doc['raw_text']
    entities = doc.get('entities', [])

    if not text.strip():
        return None

    # Step 1: word spans
    word_spans = [(m.start(), m.end(), m.group())
                  for m in re.finditer(r'\S+', text)]

    if not word_spans:
        return None

    words = [w for _, _, w in word_spans]
    word_labels = ['O'] * len(word_spans)

    # Step 2: assign BIO labels PROPERLY
    VALID_ENTITY_TYPES = {'Pest','Plant','Disease','Vector','Location','Dissemination_pathway'}

    for ent in entities:
        if ent.get('type') not in VALID_ENTITY_TYPES:
            continue
        es = ent.get('start')
        ee = ent.get('end')
        et = ent.get('type')

        if et not in ENTITY_TYPES:
            continue
        if es is None or ee is None:
            continue
        if not (0 <= es < ee <= len(text)):
            continue

        started = False

        for i, (ws, we, _) in enumerate(word_spans):
            # overlap
            if not (we <= es or ws >= ee):
                if not started:
                    word_labels[i] = f'B-{et}'
                    started = True
                else:
                    word_labels[i] = f'I-{et}'
    if sum(1 for l in word_labels if l != 'O') < 1:
      return None

    empty = sum(1 for d in train_docs if len(d['entities']) == 0)
    print("Docs with 0 entities:", empty, "/", len(train_docs))

    # Step 3: convert to IDs
    return {
        'tokens': words,
        'ner_tags': [LABEL2ID.get(lbl, 0) for lbl in word_labels]
    }

def verify_silver_labels(docs, tokenizer, n_sample=5):
    """
    Diagnostic: shows how many non-O labels are in the silver data.
    If everything is O (label_id=0), NER training will learn nothing.
    """
    total_tokens = 0
    nonO_tokens  = 0
    for doc in docs:
        hf = doc_to_hf_fixed(doc, tokenizer)
        if not hf: continue
        total_tokens += len(hf['ner_tags'])
        nonO_tokens  += sum(1 for t in hf['ner_tags'] if t != 0)

    pct = 100 * nonO_tokens / max(1, total_tokens)
    print(f'Label stats: {total_tokens} tokens | {nonO_tokens} non-O ({pct:.2f}%)')
    if pct < 0.1:
        print('  ❌ CRITICAL: Almost no entity labels! Check silver labeling.')
        print('     Likely cause: entity text not found in doc text.')
    elif pct < 1.0:
        print('  ⚠️  Low entity density — NER will train but may underfit')
    else:
        print('  ✅ Entity density looks reasonable')

    # Show a few examples
    print(f'\nSample labeled words (showing non-O only):')
    shown = 0
    for doc in docs[:n_sample]:
        hf = doc_to_hf_fixed(doc, tokenizer)
        if not hf: continue
        for word, tag_id in zip(hf['tokens'], hf['ner_tags']):
            if tag_id != 0:
                print(f'  {word!r:30s} → {ID2LABEL[tag_id]}')
                shown += 1
                if shown >= 20: break
        if shown >= 20: break
    if shown == 0:
        print('  (none found — silver labels produced 0 entity tokens)')


class PestNERModel:

    def __init__(self, base_model=BASE_NER_MODEL, max_length=512):
        self.max_length = max_length
        self.tokenizer  = AutoTokenizer.from_pretrained(base_model)
        self.model      = AutoModelForTokenClassification.from_pretrained(
            base_model, num_labels=len(LABEL_LIST),
            id2label=ID2LABEL, label2id=LABEL2ID,
            ignore_mismatched_sizes=True,
        ).to(DEVICE)
        print(f'[NER] {base_model} | {len(LABEL_LIST)} labels | {DEVICE}')

    @classmethod
    def from_pretrained(cls, model_dir):
        obj = cls.__new__(cls); obj.max_length = 512
        obj.tokenizer = AutoTokenizer.from_pretrained(str(model_dir))
        obj.model     = AutoModelForTokenClassification.from_pretrained(
            str(model_dir)).to(DEVICE)
        print(f'[NER] Loaded from {model_dir}')
        return obj

    def _tokenize_and_align(self, examples):
        """
        Tokenize a batch of word-tokenized sentences and align BIO labels.

        Key correctness invariant:
          - First subword of word i  → label = ner_tags[i]
          - Continuation subwords    → label = -100  (ignored by loss)
          - Special tokens [CLS][SEP]→ label = -100
        """
        tok = self.tokenizer(
            examples['tokens'],
            truncation=True,
            max_length=self.max_length,
            is_split_into_words=True,
            padding=False,
        )
        all_labels = []
        for i, word_labels in enumerate(examples['ner_tags']):
            word_ids = tok.word_ids(batch_index=i)
            prev_wid = None
            aligned  = []
            for wid in word_ids:
                if wid is None:
                    # [CLS], [SEP], padding
                    aligned.append(-100)
                elif wid != prev_wid:
                    # First subword of this word → real label
                    aligned.append(word_labels[wid])
                else:
                    # Continuation subword → ignore
                    aligned.append(-100)
                prev_wid = wid
            all_labels.append(aligned)
        tok['labels'] = all_labels
        return tok

    def train(self, train_docs, eval_docs, output_dir,
              epochs=8, batch_size=16, lr=3e-5):
        # Convert docs → HF format using the fixed function
        train_hf = []
        for d in train_docs:
            if d.get('entities'):
                x = doc_to_hf_fixed(d, self.tokenizer)
                if x:
                    train_hf.append(x)
        train_hf = [x for x in train_hf if x]  # remove None

        eval_hf  = [doc_to_hf_fixed(d, self.tokenizer)
                    for d in eval_docs if d.get('entities')]
        eval_hf  = [x for x in eval_hf if x]

        if not train_hf:
            print('[NER] ❌ No training examples — run silver labeling first'); return

        # ── Sanity check before spending GPU time ────────────────────────────
        all_tags    = [t for ex in train_hf for t in ex['ner_tags']]
        nonO        = sum(1 for t in all_tags if t != 0)
        pct         = 100 * nonO / max(1, len(all_tags))
        print(f'[NER] Train: {len(train_hf)} docs | '
              f'{len(all_tags)} tokens | {nonO} entity tokens ({pct:.2f}%)')
        if pct < 0.05:
            print('[NER] ❌ ABORTING — entity label rate is too low (<0.05%).')
            print('         Fix silver labeling before training.')
            return

        train_ds = Dataset.from_list(train_hf).map(self._tokenize_and_align, batched=True)
        eval_ds  = Dataset.from_list(eval_hf).map(self._tokenize_and_align,  batched=True)
        total_steps = (len(train_ds) // batch_size) * epochs
        warmup_steps = int(0.1 * total_steps)
        args = TrainingArguments(
            output_dir=str(output_dir),
            num_train_epochs=epochs,
            per_device_train_batch_size=batch_size,
            per_device_eval_batch_size=batch_size,
            learning_rate=lr,
            weight_decay=0.01,
            warmup_steps=warmup_steps,

            eval_strategy='epoch',
            save_strategy='epoch',
            load_best_model_at_end=True,
            metric_for_best_model='eval_f1',

            fp16=(DEVICE == 'cuda'),
            report_to='none',
            logging_steps=20,
        )
        class_weights_ner = build_ner_class_weights(train_hf, LABEL_LIST,max_weight=5.0)

        trainer = WeightedTrainer(
            model=self.model,
            args=args,
            train_dataset=train_ds,
            eval_dataset=eval_ds,
            processing_class=self.tokenizer,
            data_collator=DataCollatorForTokenClassification(self.tokenizer),
            compute_metrics=self._compute_metrics,
            class_weights=class_weights_ner
        )
        trainer.train()
        trainer.save_model(str(output_dir))
        self.tokenizer.save_pretrained(str(output_dir))
        print(f'[NER] ✅ Saved → {output_dir}')


    def _compute_metrics(self, p):
      predictions, labels = p
      predictions = predictions.argmax(axis=2)

      true_predictions = []
      true_labels = []

      for pred, lab in zip(predictions, labels):
          curr_preds = []
          curr_labels = []

          for p_i, l_i in zip(pred, lab):
              if l_i != -100:
                  curr_preds.append(ID2LABEL[p_i])
                  curr_labels.append(ID2LABEL[l_i])

          true_predictions.append(curr_preds)
          true_labels.append(curr_labels)

      return {
          "eval_precision": precision_score(true_labels, true_predictions),
          "eval_recall": recall_score(true_labels, true_predictions),
          "eval_f1": f1_score(true_labels, true_predictions),
      }
    def predict(self, text):
        self.model.eval()
        enc     = self.tokenizer(text, return_offsets_mapping=True,
                                 add_special_tokens=True, truncation=False)
        ids     = enc['input_ids']
        offsets = enc['offset_mapping']
        stride  = self.max_length - 50
        all_lid = [0]   * len(ids)
        all_sc  = [0.0] * len(ids)
        for start in range(0, len(ids), stride):
            end  = min(start + self.max_length, len(ids))
            inp  = torch.tensor([ids[start:end]]).to(DEVICE)
            with torch.no_grad():
                logits = self.model(inp).logits
            probs = torch.softmax(logits, dim=-1)[0]
            preds = torch.argmax(probs, dim=-1)
            for i in range(end - start):
                if start + i < len(all_lid):
                    all_lid[start + i] = preds[i].item()
                    all_sc[start + i]  = probs[i, preds[i]].item()
        return self._decode(text, offsets, all_lid, all_sc)

    def _decode(self, text, offsets, lids, scores):
      ents = []
      ct = cs = ce = None
      csc = 0.0

      for (s, e), lid, sc in zip(offsets, lids, scores):
          lbl = ID2LABEL[lid]

          if lbl.startswith('B-'):
              # 🔁 close previous entity
              if ct:
                  sp = text[cs:ce].strip()
                  if sp and csc >= 0.7 and len(sp)>=4:   # 🔥 ADDED FILTER
                      ents.append({
                          'text': sp,
                          'type': ct,
                          'start': cs,
                          'end': ce,
                          'score': csc,
                          'canonical_id': ''
                      })

              ct, cs, ce, csc = lbl[2:], s, e, sc

          elif lbl.startswith('I-') and lbl[2:] == ct:
              ce = e
              csc = min(csc, sc)

          else:
              # 🔁 close entity on break
              if ct:
                  sp = text[cs:ce].strip()
                  if sp and csc >= 0.5:   # 🔥 ADDED FILTER
                      ents.append({
                          'text': sp,
                          'type': ct,
                          'start': cs,
                          'end': ce,
                          'score': csc,
                          'canonical_id': ''
                      })

              ct = cs = ce = None
              csc = 0.0

      # 🔁 final entity
      if ct:
          sp = text[cs:ce].strip()
          if sp and csc >= 0.5:   # 🔥 ADDED FILTER
              ents.append({
                  'text': sp,
                  'type': ct,
                  'start': cs,
                  'end': ce,
                  'score': csc,
                  'canonical_id': ''
              })

      return ents


print('✅ NER model class ready (fixed tokenization)')


## Relation model (unchanged — was working)

In [ ]:
def clean_entity(e: str):
    if e is None:
        return None

    e = e.strip()

    fixes = {
        "itrus": "citrus",
        "yle": None,
        "South": None,
        "countries": None,
        "international": None,
        "American": "USA",
        "European": "Europe"
    }

    if e in fixes:
        return fixes[e]

    # remove broken tokens
    if len(e) <= 1:
        return None

    return e

In [ ]:
def is_valid_relation(s, o):
    invalid = {"countries", "international", "state", "country"}

    if s is None or o is None:
        return False

    if s in invalid or o in invalid:
        return False

    if s.lower() == o.lower():
        return False

    return True

In [ ]:
def mine_hard_negatives(entities, pos_pairs, max_per_pos=3):
    hard_negs = []

    for s, o in itertools.permutations(entities, 2):

        if (s['text'], o['text']) in pos_pairs:
            continue

        # HARD NEGATIVE RULES
        same_sentence = True  # assume doc-level
        close_type = s['type'] == o['type']
        short_distance_bias = abs(s.get('start', 0) - o.get('start', 0)) < 200

        if (same_sentence and close_type) or short_distance_bias:
            hard_negs.append((s, o))

    random.shuffle(hard_negs)
    return hard_negs

In [ ]:
SUBJ_START, SUBJ_END = '[SUBJ_START]', '[SUBJ_END]'
OBJ_START,  OBJ_END  = '[OBJ_START]',  '[OBJ_END]'
SPECIAL_TOKENS = [SUBJ_START, SUBJ_END, OBJ_START, OBJ_END]

class RelationClassifier(nn.Module):
    def __init__(self, encoder_name=BASE_NER_MODEL, num_labels=len(PREDICATES), dropout=0.15):
        super().__init__()
        cfg = AutoConfig.from_pretrained(encoder_name)
        self.encoder = AutoModel.from_pretrained(encoder_name)
        H = cfg.hidden_size
        self.classifier = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(H*2 + 3, H),   # +1 for distance
            nn.GELU(),
            nn.LayerNorm(H),
            nn.Dropout(dropout),
            nn.Linear(H, num_labels),
        )

    def forward(self, input_ids, attention_mask, subj_pos, obj_pos, distance, subj_norm, obj_norm):
        h = self.encoder(
            input_ids=input_ids,
            attention_mask=attention_mask
        ).last_hidden_state

        B = h.size(0)

        pooled = torch.cat([
            h[torch.arange(B), subj_pos],
            h[torch.arange(B), obj_pos],
            distance.unsqueeze(1),
            subj_norm.unsqueeze(1),
            obj_norm.unsqueeze(1)
        ], dim=-1)

        logits = self.classifier(pooled)
        return logits

@dataclass
class RelExample:
    doc_id:str; text:str
    subj_text:str; subj_type:str; subj_start:int; subj_end:int
    obj_text:str;  obj_type:str;  obj_start:int;  obj_end:int
    label:int=0

def insert_markers(text,ss,se,os_,oe):
    spans=sorted([(ss,SUBJ_START),(se,SUBJ_END),(os_,OBJ_START),(oe,OBJ_END)],
                 key=lambda x:x[0],reverse=True)
    for pos,marker in spans:
        pos=max(0,min(pos,len(text)))
        text=text[:pos]+f' {marker} '+text[pos:]
    return text

class RelationModel:
    def __init__(self, base_model=BASE_NER_MODEL, max_length=512):
        self.max_length=max_length
        self.tokenizer=AutoTokenizer.from_pretrained(base_model)
        self.tokenizer.add_tokens(SPECIAL_TOKENS,special_tokens=True)
        self.model=RelationClassifier(encoder_name=base_model).to(DEVICE)
        self.model.encoder.resize_token_embeddings(len(self.tokenizer))
        print(f'[REL] {base_model} | {len(PREDICATES)} predicates | {DEVICE}')

    @classmethod
    def from_pretrained(cls,d):
        o=cls.__new__(cls); o.max_length=512
        o.tokenizer=AutoTokenizer.from_pretrained(str(d))
        o.model=torch.load(Path(d)/'relation_model.pt',map_location=DEVICE)
        o.model.eval(); print(f'[REL] Loaded from {d}'); return o

    def save(self,d):
        Path(d).mkdir(parents=True,exist_ok=True)
        torch.save(self.model,Path(d)/'relation_model.pt')
        self.tokenizer.save_pretrained(str(d))
        print(f'[REL] ✅ Saved → {d}')

    def _encode(self, ex):
        marked = insert_markers(
            ex.text,
            ex.subj_start, ex.subj_end,
            ex.obj_start, ex.obj_end
        )

        enc = self.tokenizer(
            marked,
            max_length=self.max_length,
            truncation=True,
            padding='max_length',
            return_tensors='pt'
        )

        ids = enc['input_ids'][0]

        sid = self.tokenizer.convert_tokens_to_ids(SUBJ_START)
        oid = self.tokenizer.convert_tokens_to_ids(OBJ_START)

        sp = (ids == sid).nonzero(as_tuple=True)[0]
        op = (ids == oid).nonzero(as_tuple=True)[0]

        subj_pos = sp[0].item() if len(sp) else 1
        obj_pos  = op[0].item() if len(op) else 2

        subj_norm = subj_pos / self.max_length
        obj_norm  = obj_pos / self.max_length

        # 🔥 NEW: distance feature
        distance = abs(subj_pos - obj_pos)
        distance = min(distance, 512) / 512.0

        #WOW
        return {
            'input_ids': ids,
            'attention_mask': enc['attention_mask'][0],
            'subj_marker_pos': torch.tensor(subj_pos),
            'obj_marker_pos': torch.tensor(obj_pos),
            'distance': torch.tensor(distance, dtype=torch.float),
            'subj_norm': torch.tensor(subj_norm, dtype=torch.float),
            'obj_norm': torch.tensor(obj_norm, dtype=torch.float),
            'labels': torch.tensor(ex.label)
        }

    def train(self,train_ex,eval_ex,output_dir,epochs=5,batch_size=16,lr=2e-5,no_rel_w=0.25):
        w=torch.ones(len(PREDICATES)).to(DEVICE); w[NO_REL_ID]=no_rel_w
        class_weights = torch.ones(len(PREDICATES)).to(DEVICE)
        class_weights[NO_REL_ID] = no_rel_w
        # === CLASS WEIGHT FIX (LOG SMOOTHING) ===

        class_counts = torch.bincount(
            torch.tensor([e.label for e in train_ex]),
            minlength=len(PREDICATES)
        ).float()

        total = class_counts.sum()

        class_weights = torch.log1p(total / (class_counts + 1))

        # safety clamps (VERY IMPORTANT)
        class_weights = torch.clamp(class_weights, min=0.5, max=10.0)

        class_weights = class_weights.to(DEVICE)

        print(f"[REL] Fixed class weights: {class_weights.tolist()}")

        loss_fn = nn.CrossEntropyLoss(weight=class_weights)
        opt=AdamW(self.model.parameters(),lr=lr,weight_decay=0.01)
        steps=max(1,len(train_ex)//batch_size)*epochs
        sched=get_linear_schedule_with_warmup(opt,int(0.1*steps),steps)
        print(f'[REL] Encoding {len(train_ex)} train / {len(eval_ex)} eval...')
        tr_enc=[self._encode(e) for e in tqdm(train_ex,desc='Encode train')]
        ev_enc=[self._encode(e) for e in tqdm(eval_ex, desc='Encode eval')]
        best=0.0
        for ep in range(epochs):
            self.model.train(); total=0; n=0
            idx=torch.randperm(len(tr_enc))
            for start in range(0,len(tr_enc),batch_size):
                b=[tr_enc[i] for i in idx[start:start+batch_size]]
                opt.zero_grad()
                logits = self.model(
                    torch.stack([x['input_ids'] for x in b]).to(DEVICE),
                    torch.stack([x['attention_mask'] for x in b]).to(DEVICE),
                    torch.stack([x['subj_marker_pos'] for x in b]).to(DEVICE),
                    torch.stack([x['obj_marker_pos'] for x in b]).to(DEVICE),
                    torch.stack([x['distance'] for x in b]).to(DEVICE),
                    torch.stack([x['subj_norm'] for x in b]).to(DEVICE),
                    torch.stack([x['obj_norm'] for x in b]).to(DEVICE),
                )

                labels = torch.stack([x['labels'] for x in b]).to(DEVICE)

                loss = loss_fn(logits, labels)
                loss.backward()
                nn.utils.clip_grad_norm_(self.model.parameters(),1.0)
                opt.step(); sched.step(); total+=loss.item(); n+=1
            f1=self._eval(ev_enc,batch_size)
            print(f'  Epoch {ep+1}/{epochs}  loss={total/max(1,n):.4f}  F1={f1:.4f}')
            if f1>best: best=f1; self.save(output_dir)
        print(f'[REL] Best F1: {best:.4f}')

    def _eval(self, enc, bs=16, k=3):
        self.model.eval()

        preds = []
        labels = []

        with torch.no_grad():
            for start in range(0, len(enc), bs):
                b = enc[start:start+bs]

                logits = self.model(
                    torch.stack([x['input_ids'] for x in b]).to(DEVICE),
                    torch.stack([x['attention_mask'] for x in b]).to(DEVICE),
                    torch.stack([x['subj_marker_pos'] for x in b]).to(DEVICE),
                    torch.stack([x['obj_marker_pos'] for x in b]).to(DEVICE),
                    torch.stack([x['distance'] for x in b]).to(DEVICE),
                    torch.stack([x['subj_norm'] for x in b]).to(DEVICE),
                    torch.stack([x['obj_norm'] for x in b]).to(DEVICE),
                )

                topk = torch.topk(logits, k=k, dim=-1).indices.cpu().tolist()
                batch_labels = [x['labels'].item() for x in b]

                for i, true in enumerate(batch_labels):
                    preds.append(topk[i][0])

                labels.extend(batch_labels)

        pos = [i for i in range(len(PREDICATES)) if i != NO_REL_ID]

        return sk_f1(labels, preds, labels=pos, average='macro', zero_division=0)

    def predict_document(self, text, entities, threshold=0.45):
        self.model.eval()
        rels = []

        for s, o in itertools.permutations(entities, 2):

            if (s['type'], o['type']) not in VALID_PAIRS:
                continue

            ex = RelExample(
                'inf', text,
                s['text'], s['type'], s.get('start', 0), s.get('end', 0),
                o['text'], o['type'], o.get('start', 0), o.get('end', 0)
            )

            enc = self._encode(ex)

            with torch.no_grad():
                logits = self.model(
                    enc['input_ids'].unsqueeze(0).to(DEVICE),
                    enc['attention_mask'].unsqueeze(0).to(DEVICE),
                    enc['subj_marker_pos'].unsqueeze(0).to(DEVICE),
                    enc['obj_marker_pos'].unsqueeze(0).to(DEVICE),
                    enc['distance'].unsqueeze(0).to(DEVICE),
                    enc['subj_norm'].unsqueeze(0).to(DEVICE),
                    enc['obj_norm'].unsqueeze(0).to(DEVICE),
                )

                probs = torch.softmax(logits, dim=-1)[0]
                pid = torch.argmax(probs).item()
                pred = ID2PRED[pid]
                score = probs[pid].item()

            if pred != 'NO_RELATION' and score >= threshold:
                rels.append({
                    'subject': s.get('canonical_id') or s['text'],
                    'predicate': pred,
                    'object': o.get('canonical_id') or o['text'],
                    'score': score
                })

        return rels

    def predict_single(self, text, e1, e2):
        self.model.eval()

        ex = RelExample(
            doc_id="inf",
            text=text,
            subj_text=e1['text'],
            subj_type=e1['type'],
            subj_start=e1.get('start', 0),
            subj_end=e1.get('end', 0),
            obj_text=e2['text'],
            obj_type=e2['type'],
            obj_start=e2.get('start', 0),
            obj_end=e2.get('end', 0),
            label=0
        )

        enc = self._encode(ex)

        with torch.no_grad():
            logits = self.model(
                enc['input_ids'].unsqueeze(0).to(DEVICE),
                enc['attention_mask'].unsqueeze(0).to(DEVICE),
                enc['subj_marker_pos'].unsqueeze(0).to(DEVICE),
                enc['obj_marker_pos'].unsqueeze(0).to(DEVICE),
                enc['distance'].unsqueeze(0).to(DEVICE),
                enc['subj_norm'].unsqueeze(0).to(DEVICE),
                enc['obj_norm'].unsqueeze(0).to(DEVICE),
            )

            probs = torch.softmax(logits, dim=-1)[0]
            pid = torch.argmax(probs).item()

            return ID2PRED[pid], probs[pid].item()


def build_relation_examples(docs, neg_ratio=3.0):
    examples = []

    for doc in docs:
        text = doc['raw_text']
        ents = doc.get('entities', [])

        em = {e['text']: e for e in ents}
        pos_set = set()

        positives = []

        # POSITIVES
        for r in doc.get('relations', []):
            s = em.get(r.get('subject', ''))
            o = em.get(r.get('object', ''))
            p = r.get('predicate', '')

            if s and o and p in PRED2ID and PRED2ID[p] != NO_REL_ID:
                ex = RelExample(
                    doc['id'], text,
                    s['text'], s['type'], s.get('start', 0), s.get('end', 0),
                    o['text'], o['type'], o.get('start', 0), o.get('end', 0),
                    label=PRED2ID[p]
                )
                examples.append(ex)
                positives.append((s, o))
                pos_set.add((s['text'], o['text']))

        # HARD NEGATIVES (IMPROVED)
        hard_negs = []

        for s, o in itertools.permutations(ents, 2):

            if (s['text'], o['text']) in pos_set:
                continue

            if (s['type'], o['type']) not in VALID_PAIRS:
                continue

            # 🔥 HARD FILTER 1: distance-based closeness
            dist = abs(s.get('start', 0) - o.get('start', 0))
            if dist > 200:
                continue

            # 🔥 HARD FILTER 2: same entity neighborhood boost
            if s['type'] == o['type']:
                dist = dist * 0.7

            # 🔥 HARD FILTER 3: avoid trivial pairs
            if s['text'].lower() == o['text'].lower():
                continue

            hard_negs.append((s, o))

        random.shuffle(hard_negs)

        # LIMIT NEGATIVES
        for s, o in hard_negs[:int(len(positives) * neg_ratio)]:
            examples.append(RelExample(
                doc['id'], text,
                s['text'], s['type'], s.get('start', 0), s.get('end', 0),
                o['text'], o['type'], o.get('start', 0), o.get('end', 0),
                label=NO_REL_ID
            ))

    pos = sum(1 for e in examples if e.label != NO_REL_ID)

    print(f'✅ {len(examples)} examples: {pos} positive | {len(examples)-pos} negative')

    return examples

print('✅ Relation model class ready')

## KG assembly & Evaluation

In [ ]:
def norm_edge(s,p,o): return (s.strip(), predicate_normalize(p), o.strip())

def build_kg(entities, relations, score_threshold=0.35):
    em={}
    for e in entities:
        cid=e.get('canonical_id') or e['text']
        em[e['text']]=cid; em[cid]=cid
    edges,seen=[],set()
    for r in sorted(relations,key=lambda x:-x.get('score',0)):
        s_raw = r.get('subject','')
        o_raw = r.get('object','')

        s_clean = clean_entity(s_raw)
        o_clean = clean_entity(o_raw)

        if not is_valid_relation(s_clean, o_clean):
            continue
        if r.get('score',1)<score_threshold: continue
        s = em.get(s_clean)
        o = em.get(o_clean)
        k=norm_edge(s,r['predicate'],o)
        if k not in seen: seen.add(k); edges.append({'subject':s,'predicate':r['predicate'],'object':o})
    return edges

def doc_f1(pred,ref):
    if not ref and not pred: return 1.0
    if not ref or not pred:  return 0.0
    pc=Counter(norm_edge(*e) for e in pred)
    rc=Counter(norm_edge(*e) for e in ref)
    tp=sum((pc&rc).values())
    P=tp/len(pred); R=tp/len(ref)
    return 2*P*R/(P+R) if (P+R)>0 else 0.0

def evaluate(predictions, references, verbose=False):
    per_doc={}
    for doc_id,ref in references.items():
        pred=predictions.get(doc_id,[])
        f1=doc_f1(pred,ref); per_doc[doc_id]=f1
        if verbose:
            icon='✓' if f1>=0.5 else ('~' if f1>0 else '✗')
            print(f'  {icon} {doc_id:<35} F1={f1:.4f}  pred={len(pred):<4} ref={len(ref)}')
    macro=sum(per_doc.values())/len(per_doc) if per_doc else 0.0
    print(f'\n  MACRO F1 = {macro:.4f}  ({len(per_doc)} docs)')
    return {'macro_f1':round(macro,6),'per_doc':per_doc}

def format_submission(submission, output_path):
    rows=[]
    for doc_id,edges in submission.items():
        kg=json.dumps([{'predicate':e['predicate'],'subject':e['subject'],'object':e['object']}
                       for e in edges],ensure_ascii=False)
        rows.append({'doc_id':doc_id,'knowledge_graph':kg})
    with open(output_path,'w',newline='',encoding='utf-8') as f:
        w=csv.DictWriter(f,fieldnames=['doc_id','knowledge_graph'],quoting=csv.QUOTE_ALL)
        w.writeheader(); w.writerows(rows)
    total=sum(len(v) for v in submission.values())
    print(f'✅ Submission: {len(submission)} docs | {total} edges → {output_path}')

print('✅ KG assembly and evaluation ready')

## Step 1 — Load data & silver label

Run these cells in order.

In [ ]:
train_docs = load_split(TRAIN_DIR)
dev_docs   = load_split(DEV_DIR)
test_docs  = load_split(TEST_DIR)

In [ ]:
# Silver labeling — uses LLM if key is set, falls back to rule-based
print('Silver labeling train...')
for doc in tqdm(train_docs, desc='Train'):
    high_quality_silver_label_doc(doc)

print('Silver labeling dev...')
for doc in tqdm(dev_docs, desc='Dev'):
    high_quality_silver_label_doc(doc)

total_ents = sum(len(d.get('entities',[])) for d in train_docs)
total_rels = sum(len(d.get('relations',[])) for d in train_docs)
print(f'\nTrain silver: {total_ents} entities | {total_rels} relations')

In [ ]:
#Debug
for doc in train_docs[:3]:
    print("\nTEXT:", doc['raw_text'][:200])
    print("ENTITIES:", doc['entities'][:10])

## Step 2 — Diagnose labels before training

**Run this cell before training.** If it shows 0% non-O labels, your silver labeling produced empty annotations — do NOT train yet, fix the silver labels first.

In [ ]:
# Load a tokenizer just for diagnosis (no model needed yet)
_diag_tok = AutoTokenizer.from_pretrained(BASE_NER_MODEL)
print('=== Train set label diagnosis ===')
verify_silver_labels(train_docs, _diag_tok, n_sample=5)
print('\n=== Dev set label diagnosis ===')
verify_silver_labels(dev_docs, _diag_tok, n_sample=3)

## Step 3 — Train NER

Only run after the diagnosis above shows >0.1% non-O labels.

In [ ]:
train_hf = [
    doc_to_hf_fixed(d, _diag_tok)
    for d in train_docs
    if d.get('entities')
]

train_hf = [x for x in train_hf if x]

In [ ]:
import random

for ex in random.sample(train_hf, 10):
    print(list(zip(
        ex["tokens"][:20],
        [ID2LABEL[t] for t in ex["ner_tags"][:20]]
    )))
    print()

In [ ]:
def build_ner_class_weights(train_hf, label_list, max_weight=5.0):
    """
    Compute stable class weights for NER.

    Key principles:
      - O label gets weight 1.0 (it's the majority, no boost needed)
      - Entity labels get boosted by inverse frequency, capped at max_weight
      - Labels with zero occurrences get weight 1.0 (not 0.0) so the model
        can still predict them if it sees them at test time
    """
    label_counts = Counter()
    for ex in train_hf:
        for t in ex['ner_tags']:
            if t != -100:
                label_counts[t] += 1

    total = sum(label_counts.values())
    num_classes = len(label_list)

    weights = []
    for i in range(num_classes):
        label = label_list[i]
        freq  = label_counts.get(i, 0)

        if label == 'O':
            # O is majority class — keep at 1.0, never boost it
            weights.append(1.0)
        elif freq == 0:
            # Label never seen in training data.
            # Don't give 0.0 (model ignores it) or huge value (explodes).
            # Give 1.0 so loss is computed normally if it appears.
            weights.append(1.0)
        else:
            # Inverse frequency, capped
            raw = (total / (num_classes * freq))
            weights.append(min(raw, max_weight))

    class_weights = torch.tensor(weights, dtype=torch.float)

    print(f"[NER] Class weights (capped at {max_weight}):")
    for i, (lbl, w) in enumerate(zip(label_list, weights)):
        bar = '█' * int(w)
        print(f"  {lbl:<30} {w:5.2f}  {bar}")

    return class_weights

In [ ]:
def build_relation_class_weights(num_relations, no_rel_id, no_rel_weight=0.25):
    """
    Creates class weights for relation classification.
    Keeps NO_RELATION downweighted.
    """

    weights = torch.ones(num_relations, dtype=torch.float)
    weights[no_rel_id] = no_rel_weight

    print(f"[REL] Class weights: {weights.tolist()}")
    return weights

In [ ]:
from collections import Counter
import torch

label_counts = Counter()

for ex in train_hf:
    for t in ex["ner_tags"]:
        if t != -100:
            label_counts[t] += 1

total = sum(label_counts.values())

weights = []
for i in range(len(LABEL_LIST)):
    freq = label_counts.get(i, 0)
    if freq == 0:
        weights.append(0.0)
    else:
        weights.append(total / (len(LABEL_LIST) * freq))

class_weights = torch.tensor(weights, dtype=torch.float)

In [ ]:
from transformers import Trainer
import torch
import torch.nn as nn

class WeightedTrainer(Trainer):
    def __init__(self, *args, class_weights=None, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.get("labels")
        outputs = model(**inputs)
        logits = outputs.get("logits")

        loss_fct = nn.CrossEntropyLoss(
            weight=self.class_weights.to(logits.device) if self.class_weights is not None else None,
            ignore_index=-100
        )

        loss = loss_fct(
            logits.view(-1, logits.shape[-1]),
            labels.view(-1)
        )

        return (loss, outputs) if return_outputs else loss

In [ ]:
ner_model = PestNERModel()
ner_model.train(train_docs, dev_docs, NER_MODEL_DIR)


## Step 4 — Train relation model

In [ ]:
examples = build_relation_examples(train_docs + dev_docs, neg_ratio=3.0)
random.shuffle(examples)
val_n     = max(1, int(len(examples) * 0.15))
rel_model = RelationModel()
rel_model.train(examples[val_n:], examples[:val_n], REL_MODEL_DIR, epochs=5)

## Step 5 — Run pipeline on test set

In [ ]:
print("Sample NER output:")
print(ner_model.predict(test_docs[0]['raw_text'])[:5])

In [ ]:
def generate_pairs(entities, max_pairs=30):
    """
    Smart pair generation to reduce noise
    """
    pairs = []

    for i in range(len(entities)):
        for j in range(len(entities)):
            if i == j:
                continue

            e1 = entities[i]
            e2 = entities[j]

            # ❌ skip same text
            if e1['text'].lower() == e2['text'].lower():
                continue

            # ❌ avoid same type (very noisy)
            if e1['type'] == e2['type']:
                continue

            # ✅ enforce structure: subject → Location
            if e2['type'] != 'Location':
                continue

            pairs.append((e1, e2))

    return pairs[:max_pairs]

In [ ]:
def run_pipeline(docs, ner_model, rel_model, rel_threshold=0.25, kg_threshold=0.2):
    submission = {}

    VALID_ENTITY_TYPES = {
        'Pest','Plant','Disease','Vector',
        'Location','Dissemination_pathway'
    }

    BAD_WORDS = {"global", "world"}

    for doc in tqdm(docs, desc='Pipeline'):
        try:
            text = doc['raw_text']

            # 🔹 STEP 1: NER
            ents = ner_model.predict(text)

            # 🔹 STEP 2: FILTER ENTITIES (balanced, not too aggressive)
            filtered = []
            for e in ents:
                t = e['text'].strip()
                if not t:
                    continue

                if e['type'] not in VALID_ENTITY_TYPES:
                    continue

                if len(t) < 3:
                    continue

                if e.get('score', 0) < 0.45:   # slightly relaxed
                    continue

                if t.isdigit():
                    continue

                if t.lower() in BAD_WORDS:
                    continue

                filtered.append(e)

            # 🔹 STEP 3: DEDUPLICATION
            seen = set()
            ents_list = []

            for e in filtered:
                key = (e['text'].lower(), e['type'])
                if key in seen:
                    continue
                seen.add(key)

                e['canonical_id'] = e['text']
                ents_list.append(e)

            # limit size (avoid explosion)
            ents_list = ents_list[:50]

            # early exit
            if len(ents_list) < 2:
                submission[doc['id']] = []
                continue

            # 🔹 STEP 4: RELATION EXTRACTION (single clean call)
            rels = rel_model.predict_document(
                text,
                ents_list,
                threshold=rel_threshold
            )

            # 🔹 STEP 5: LIGHT CLEANING (DO NOT OVER-FILTER)
            clean_rels = []
            seen_rel = set()

            for r in rels:
                subj = r['subject'].lower()
                obj  = r['object'].lower()
                pred = r['predicate']

                # skip trivial bad objects
                if obj in BAD_WORDS or obj.isdigit():
                    continue

                # deduplicate only
                key = (pred, subj, obj)
                if key in seen_rel:
                    continue

                seen_rel.add(key)
                clean_rels.append(r)

            # 🔹 STEP 6: BUILD KNOWLEDGE GRAPH
            edges = build_kg(ents_list, clean_rels, kg_threshold)

            submission[doc['id']] = edges

        except Exception as ex:
            print(f'[ERROR] {doc["id"]}: {ex}')
            submission[doc['id']] = []

    total = sum(len(v) for v in submission.values())
    print(f'✅ {len(submission)} docs | {total} total edges | avg {total/max(1,len(submission)):.2f}/doc')

    return submission

In [ ]:
test_sub = run_pipeline(test_docs, ner_model, rel_model)

# Save output
format_submission(test_sub, SUBMISSION_DIR / 'submission.csv')

In [ ]:
import json
import datetime
from pathlib import Path

def log_results(submission, docs, output_path):
    """
    Logs pipeline statistics + saves summary report.
    """

    total_docs = len(submission)
    total_edges = sum(len(v) for v in submission.values())
    avg_edges = total_edges / max(1, total_docs)

    empty_docs = sum(1 for v in submission.values() if len(v) == 0)

    report = {
        "timestamp": str(datetime.datetime.now()),
        "total_documents": total_docs,
        "total_edges": total_edges,
        "avg_edges_per_doc": round(avg_edges, 3),
        "empty_docs": empty_docs,
        "non_empty_docs": total_docs - empty_docs,
    }

    print("\n📊 ===== PIPELINE RESULTS =====")
    print(json.dumps(report, indent=4))

    # Save JSON report
    Path(output_path).parent.mkdir(parents=True, exist_ok=True)
    with open(output_path, "w") as f:
        json.dump(report, f, indent=4)

    print(f"\n✅ Saved results report → {output_path}")


# 🔥 CALL THIS AFTER PIPELINE
log_results(
    submission=test_sub,
    docs=test_docs,
    output_path=SUBMISSION_DIR / "run_report.json"
)